In [ ]:
# ============================================================
# Cellule 1 : Configuration de l'environnement et Repo
# ============================================================
import os
import shutil
import torch

# 1. Vérification du GPU (Crucial sur Colab)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"--- 🚀 Exécution sur : {device.upper()} ---")

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True  # ✅ plus rapide si les tailles d’images ne changent pas
    
if torch.cuda.is_available():
    print(f"GPU détecté : {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ ATTENTION : Le GPU n'est pas activé. Allez dans l'icône de gestion du runtime.")

# 2. Gestion du dossier projet
repo_url = "https://github.com/Lduvignacq/Deep-learning-project.git"
project_dir = "Deep-learning-project"

# Si le dossier existe, on le supprime pour repartir à neuf
if os.path.exists(project_dir):
    print(f"Suppression de l'ancien dossier {project_dir}...")
    shutil.rmtree(project_dir)

# 3. Clonage du repo
print(f"Clonage du repo : {repo_url}")
!git clone {repo_url}

# 4. Déplacement dans le projet
%cd {project_dir}

# 5. Vérification finale
print("\n--- Contenu du projet : ---")
!dir  # 'dir' fonctionne sur Windows/Colab dans VS Code pour lister les fichiers

In [ ]:
# ============================================================
# Cellule 2 : Installation des dépendances
# ============================================================

# (Vous pouvez sauter cette cellule si vous avez déjà exécuté le colab_setup
# dans CE runtime.)

!pip install --upgrade pip -q

# Installation de better-exceptions (requis par train.py souvent)
!pip install -q better-exceptions

# Installation des packages de base
!pip install -q future pandas tqdm yacs pretrainedmodels

# Installation de PyTorch et TensorBoard
!pip install -q torch torchvision tensorboard

# Installation des packages d'images
!pip install -q scikit-image imageio Pillow matplotlib Shapely scipy six opencv-python

# Installation d'albumentations
!pip install -q albumentations

print("✅ Installation terminée.")

In [ ]:
# ============================================================
# Cellule 3 : Monter Google Drive et préparer le dataset APPA-REAL
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os

# 👉 Chemin vers appa-real-release.zip dans votre Drive
zip_path = '/content/drive/MyDrive/appa-real-release.zip'

if os.path.exists(zip_path):
    print("📦 Copie et décompression du dataset depuis Google Drive...")
    !cp /content/drive/MyDrive/appa-real-release.zip .
    # -o : overwrite, -q : silencieux
    !unzip -o -q appa-real-release.zip
    # Supprime les fichiers macOS cachés
    !rm -rf __MACOSX
    print("✅ Dataset prêt !")
    !ls -la appa-real-release/
else:
    print(f"❌ Fichier non trouvé: {zip_path}")
    print("📁 Veuillez uploader appa-real-release.zip dans 'Mon Drive' d'abord.")

In [ ]:
# ============================================================
# Cellule 4 : Vérification du dataset
# ============================================================

!pwd
!ls -la
print("\n📂 Contenu de appa-real-release :")
!ls -la appa-real-release

# En option : vérification avec dataset.py si fourni dans le repo
# (Vous pouvez commenter si ça n'existe pas ou si vous ne voulez pas l'utiliser)
if os.path.exists("dataset.py"):
    print("\n🔍 Vérification avec dataset.py ...")
    !python dataset.py --data_dir appa-real-release

In [ ]:
# ============================================================
# Cellule 5 : Dataset APPA-REAL pour DEX
# ============================================================

from pathlib import Path
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
import cv2
import albumentations as A
from albumentations.pytorch import ToTensorV2

DATA_ROOT = Path("appa-real-release")
print("DATA_ROOT:", DATA_ROOT.resolve())

class AgeDataset(Dataset):
    """
    Dataset APPA-REAL pour la méthode DEX.
    On lit les CSV gt_avg_*.csv et les images associées.
    On utilise l'âge apparent moyen (apparent_age ou apparent_age_avg).
    """
    def __init__(self, csv_path, img_dir, transform=None, num_classes=101):
        self.csv_path = Path(csv_path)
        self.img_dir = Path(img_dir)
        self.transform = transform
        self.num_classes = num_classes

        if not self.csv_path.exists():
            raise FileNotFoundError(f"CSV non trouvé: {self.csv_path}")
        if not self.img_dir.exists():
            raise FileNotFoundError(f"Dossier images non trouvé: {self.img_dir}")

        self.df = pd.read_csv(self.csv_path)

        if "file_name" not in self.df.columns:
            raise ValueError(f"Colonne 'file_name' absente dans {self.csv_path}")

        # Certaines versions ont 'apparent_age', d'autres 'apparent_age_avg'
        if "apparent_age" in self.df.columns:
            self.age_col = "apparent_age"
        elif "apparent_age_avg" in self.df.columns:
            self.age_col = "apparent_age_avg"
        else:
            raise ValueError(
                f"Colonne d'âge 'apparent_age' ou 'apparent_age_avg' absente dans {self.csv_path}"
            )

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_name = row["file_name"]
        age = float(row[self.age_col])

        img_path = self.img_dir / img_name
        img = cv2.imread(str(img_path))
        if img is None:
            raise FileNotFoundError(f"Image non trouvée: {img_path}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        if self.transform is not None:
            img = self.transform(image=img)["image"]

        # DEX : classification d'âge 0..100
        age_class = int(round(age))
        age_class = max(0, min(self.num_classes - 1, age_class))

        return img, age_class, age


# Transforms
train_transform = A.Compose([
    A.Resize(224, 224),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.2),
    A.Normalize(mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

valid_transform = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

NUM_CLASSES = 101  # âges 0–100

train_csv = DATA_ROOT / "gt_avg_train.csv"
valid_csv = DATA_ROOT / "gt_avg_valid.csv"
train_dir = DATA_ROOT / "train"
valid_dir = DATA_ROOT / "valid"

train_dataset = AgeDataset(train_csv, train_dir, transform=train_transform, num_classes=NUM_CLASSES)
valid_dataset = AgeDataset(valid_csv, valid_dir, transform=valid_transform, num_classes=NUM_CLASSES)

BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"✓ Train: {len(train_dataset)} images")
print(f"✓ Valid: {len(valid_dataset)} images")

In [ ]:
# ============================================================
# Cellule 6 : Modèle DEX basé sur model.py
# ============================================================

import torch
import torch.nn as nn
import sys, os

# On est déjà dans le dossier du repo grâce à la Cellule 1
# On ajoute le répertoire courant au PYTHONPATH par sécurité
sys.path.insert(0, os.getcwd())

from model import get_model  # ⚠️ utilise le model.py du repository

class DEXModel(nn.Module):
    """
    DEX (Deep EXpectation) pour l'estimation d'âge.
    - Backbone : se_resnext50_32x4d (via get_model du repo)
    - Tête : classification 101 classes (0..100)
    - L'âge estimé est E[age] = Σ age * P(age)
    """
    def __init__(self, num_classes=101, cnn_model_name="se_resnext50_32x4d", pretrained="imagenet"):
        super().__init__()

        print(f"[DEX] Chargement du backbone {cnn_model_name} (pretrained={pretrained})")
        self.feature_extractor = get_model(
            model_name=cnn_model_name,
            num_classes=num_classes,
            pretrained=pretrained
        )

        # On suppose que model.py fournit un attribut last_linear
        dim_feats = self.feature_extractor.last_linear.in_features

        self.feature_extractor.last_linear = nn.Sequential(
            nn.Dropout(p=0.5),
            nn.Linear(dim_feats, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),
            nn.Linear(512, num_classes),
        )

        self.num_classes = num_classes

    def forward(self, x):
        logits = self.feature_extractor(x)        # [B, 101]
        probs = torch.softmax(logits, dim=1)      # distribution P(age|image)
        return logits, probs


def calculate_expected_age(probs, device):
    """
    E[age] = Σ age * P(age)
    probs : [B, num_classes]
    """
    num_classes = probs.size(1)
    age_range = torch.arange(0, num_classes, dtype=torch.float32, device=device)
    expected_ages = torch.sum(probs * age_range.unsqueeze(0), dim=1)
    return expected_ages


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AGE_BINS = NUM_CLASSES

model = DEXModel(num_classes=AGE_BINS)
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("\n✓ DEX Model created:")
print(f"  - Total params: {total_params:,}")
print(f"  - Trainable params: {trainable_params:,}")
print(f"  - Device: {device}")

In [ ]:
# ============================================================
# Cellule 7 : Boucles train/valid DEX (optimisées + AMP moderne)
# ============================================================

import torch.optim as optim
import math
from pathlib import Path
import csv
from torch import amp   # 🔥 AMP version moderne

EPOCHS = 20
LR = 1e-4
CHECKPOINT_DIR = Path("checkpoint_dex")
CHECKPOINT_DIR.mkdir(exist_ok=True)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

# 🔥 Mixed precision moderne (PyTorch 2.x)
scaler = amp.GradScaler(device="cuda" if device == "cuda" else "cpu")

def epoch_loop(loader, model, criterion, optimizer, device, train=True):

    model.train() if train else model.eval()

    running_loss = 0.0
    running_mae = 0.0
    n_samples = 0

    for imgs, age_class, age_float in loader:

        imgs = imgs.to(device, non_blocking=True)
        age_class = age_class.to(device, non_blocking=True)
        age_float = age_float.to(device, non_blocking=True)

        if train:
            optimizer.zero_grad(set_to_none=True)

        # 🔥 AMP moderne
        with amp.autocast(device_type="cuda" if device == "cuda" else "cpu"):
            logits, probs = model(imgs)
            loss = criterion(logits, age_class)
            pred_age = calculate_expected_age(probs, device)

        mae = torch.abs(pred_age - age_float).sum()

        if train:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        bs = imgs.size(0)
        running_loss += loss.item() * bs
        running_mae += mae.item()
        n_samples += bs

    avg_loss = running_loss / n_samples
    avg_mae = running_mae / n_samples
    return avg_loss, avg_mae


# Fichier métriques pour logging
metrics_path = CHECKPOINT_DIR / "dex_training_metrics.csv"
if not metrics_path.exists():
    with open(metrics_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["epoch", "train_loss", "train_mae", "val_loss", "val_mae"])

print("✓ Boucles d'entraînement prêtes (AMP moderne activé).")

In [ ]:
# ============================================================
# Cellule 8 : Entraînement du modèle DEX
# ============================================================

best_val_mae = math.inf
best_ckpt_path = CHECKPOINT_DIR / "dex_best.pth"

for epoch in range(1, EPOCHS + 1):
    print(f"\n===== Epoch {epoch}/{EPOCHS} =====")

    train_loss, train_mae = epoch_loop(train_loader, model, criterion, optimizer, device, train=True)
    val_loss, val_mae = epoch_loop(valid_loader, model, criterion, optimizer=None, device=device, train=False)

    print(f"Train - loss: {train_loss:.4f} | MAE: {train_mae:.2f}")
    print(f"Valid - loss: {val_loss:.4f} | MAE: {val_mae:.2f}")

    # Sauvegarde des métriques
    with open(metrics_path, "a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([epoch, train_loss, train_mae, val_loss, val_mae])

    # Sauvegarde du meilleur modèle sur MAE
    if val_mae < best_val_mae:
        best_val_mae = val_mae
        torch.save({
            "epoch": epoch,
            "state_dict": model.state_dict(),
            "val_mae": best_val_mae,
        }, best_ckpt_path)
        print(f"💾 Nouveau meilleur modèle sauvegardé: {best_ckpt_path} (MAE={best_val_mae:.2f})")

    # petit nettoyage mémoire GPU
    if device.type == "cuda":
        torch.cuda.empty_cache()

print("\n✅ Entraînement DEX terminé.")
print(f"Meilleur MAE (val): {best_val_mae:.2f}")
print(f"Checkpoint: {best_ckpt_path}")
print(f"Métriques: {metrics_path}")

In [ ]:
# ============================================================
# Cellule 9 : Évaluation DEX sur le set test
# ============================================================

test_csv = DATA_ROOT / "gt_avg_test.csv"
test_dir = DATA_ROOT / "test"

if test_csv.exists() and test_dir.exists():
    test_dataset = AgeDataset(test_csv, test_dir, transform=valid_transform, num_classes=NUM_CLASSES)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
    print(f"✓ Test: {len(test_dataset)} images")

    # Charger le meilleur checkpoint
    best_ckpt = CHECKPOINT_DIR / "dex_best.pth"
    if best_ckpt.exists():
        print("🔁 Chargement du checkpoint:", best_ckpt)
        ckpt = torch.load(best_ckpt, map_location=device)
        model.load_state_dict(ckpt["state_dict"])
        model.eval()

        total_mae = 0.0
        n = 0
        with torch.no_grad():
            for imgs, age_class, age_float in test_loader:
                imgs = imgs.to(device)
                age_float = age_float.to(device)

                logits, probs = model(imgs)
                pred_age = calculate_expected_age(probs, device)
                total_mae += torch.abs(pred_age - age_float).sum().item()
                n += imgs.size(0)

        print(f"✅ Test MAE: {total_mae / n:.2f} ans")
    else:
        print("❌ Aucun checkpoint dex_best.pth trouvé dans", CHECKPOINT_DIR)
else:
    print("❌ Test set introuvable (gt_avg_test.csv ou dossier test).")

In [ ]:
# ============================================================
# Cellule 10 : Courbes + 8 images "seen" + 8 images "unseen"
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import random
import torch

# ---------- 1) Courbes d'entraînement / validation ----------

if metrics_path.exists():
    df = pd.read_csv(metrics_path)
    display(df.tail())

    epochs = df["epoch"]

    plt.figure(figsize=(12, 4))

    # Loss
    plt.subplot(1, 2, 1)
    plt.plot(epochs, df["train_loss"], label="Train loss")
    plt.plot(epochs, df["val_loss"], label="Val loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Évolution de la loss (DEX)")
    plt.legend()
    plt.grid(True, alpha=0.3)

    # MAE
    plt.subplot(1, 2, 2)
    plt.plot(epochs, df["train_mae"], label="Train MAE")
    plt.plot(epochs, df["val_mae"], label="Val MAE")
    plt.xlabel("Epoch")
    plt.ylabel("MAE (années)")
    plt.title("Évolution du MAE (DEX)")
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print(f"❌ Fichier de métriques introuvable : {metrics_path}")


# ---------- 2) 8 images "seen" (train) + 8 "unseen" (test/valid) ----------

# On recharge le meilleur checkpoint si dispo
best_ckpt = CHECKPOINT_DIR / "dex_best.pth"
if best_ckpt.exists():
    print("🔁 Chargement du meilleur checkpoint pour visualisation:", best_ckpt)
    ckpt = torch.load(best_ckpt, map_location=device)
    model.load_state_dict(ckpt["state_dict"])
else:
    print("⚠️ Aucun checkpoint trouvé, on utilise le modèle actuel en mémoire.")

model.eval()

# Fonction pour "dé-normaliser" l'image pour l'affichage
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406])
IMAGENET_STD = np.array([0.229, 0.224, 0.225])

def denormalize(img_tensor):
    """
    img_tensor: torch tensor [C,H,W] normalisé avec mean/std ImageNet
    → retourne une image numpy [H,W,C] en [0,1]
    """
    img = img_tensor.cpu().numpy()
    img = np.transpose(img, (1, 2, 0))  # C,H,W -> H,W,C
    img = img * IMAGENET_STD + IMAGENET_MEAN
    img = np.clip(img, 0, 1)
    return img

# Dataset "seen" = train_dataset
seen_dataset = train_dataset

# Dataset "unseen" = test_dataset si défini, sinon valid_dataset
unseen_dataset = None
if "test_dataset" in globals():
    unseen_dataset = test_dataset
    print("📂 Unseen dataset = test_dataset")
else:
    unseen_dataset = valid_dataset
    print("📂 Unseen dataset = valid_dataset (pas de test_dataset trouvé)")

num_seen = 8
num_unseen = 8

# Indices SEEN : on prend les 8 premiers de train (fixes pour comparabilité)
max_seen = min(num_seen, len(seen_dataset))
seen_indices = list(range(max_seen))

# Indices UNSEEN : aléatoires dans unseen_dataset, à chaque exécution
all_unseen_indices = list(range(len(unseen_dataset)))
if len(all_unseen_indices) >= num_unseen:
    unseen_indices = random.sample(all_unseen_indices, k=num_unseen)
else:
    unseen_indices = all_unseen_indices  # fallback si dataset trop petit

print(f"\nIndices SEEN (train) utilisés : {seen_indices}")
print(f"Indices UNSEEN (unseen_dataset) tirés : {unseen_indices}")

total_examples = len(seen_indices) + len(unseen_indices)
rows, cols = 4, 4  # 16 cases
fig, axes = plt.subplots(rows, cols, figsize=(16, 16))
axes = axes.flatten()

for ax in axes:
    ax.axis("off")

def show_example(ax, dataset, idx, title_prefix=""):
    img, age_class, age_float = dataset[idx]  # img: tensor [C,H,W]
    true_age = float(age_float)

    img_input = img.unsqueeze(0).to(device)
    with torch.no_grad():
        logits, probs = model(img_input)
        pred_age = calculate_expected_age(probs, device)
        pred_age = float(pred_age.item())

    img_vis = denormalize(img)
    ax.imshow(img_vis)
    ax.axis("off")
    ax.set_title(f"{title_prefix}\nVrai: {true_age:.1f} / Préd.: {pred_age:.1f}", fontsize=9)

# 1ère + 2e ligne : SEEN (train)
for i, idx in enumerate(seen_indices):
    if i >= rows * cols:
        break
    ax = axes[i]
    show_example(ax, seen_dataset, idx, title_prefix=f"SEEN (train) idx={idx}")

# 3e + 4e ligne : UNSEEN (test/valid)
offset = len(seen_indices)
for j, idx in enumerate(unseen_indices):
    pos = offset + j
    if pos >= rows * cols:
        break
    ax = axes[pos]
    show_example(ax, unseen_dataset, idx, title_prefix=f"UNSEEN idx={idx}")

plt.suptitle(
    "8 images SEEN (train) vs 8 images UNSEEN (test/valid)\n"
    "Vrai âge vs prédiction DEX",
    fontsize=16
)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()